# 02 — Demand Forecasting
**ARIMA vs XGBoost vs Rolling Mean** — Uninox Houseware SKU-level demand prediction with MLflow tracking.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
from config import Paths
from knowledge.feature_store.feature_engineering import load_demand, build_sku_features, get_feature_columns
demand = load_demand()
features = build_sku_features(demand)
FEAT_COLS = get_feature_columns()
print(f'Feature matrix: {features.shape}')

## 1. Train/Test Split (time-series)

In [ ]:
# Use last 3 months as holdout
cutoff = features['period'].max() - pd.DateOffset(months=3)
train = features[features['period'] <= cutoff]
test  = features[features['period'] >  cutoff]
print(f'Train: {len(train)} rows | Test: {len(test)} rows')

## 2. XGBoost — Multi-SKU Model

In [ ]:
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
X_train, y_train = train[FEAT_COLS].values, train['net_units'].values
X_test,  y_test  = test[FEAT_COLS].values,  test['net_units'].values
model = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5,
                          subsample=0.8, random_state=42, verbosity=0)
model.fit(X_train, y_train)
preds = model.predict(X_test)
rmse  = np.sqrt(mean_squared_error(y_test, preds))
mape  = mean_absolute_percentage_error(y_test, preds) * 100
print(f'XGBoost — RMSE: {rmse:.1f}  MAPE: {mape:.1f}%')

## 3. ARIMA — Single SKU Baseline

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf
# Pick the highest-demand SKU
top_sku = demand.groupby('sku_id')['net_units'].mean().idxmax()
sku_series = demand[demand['sku_id']==top_sku].set_index('period')['net_units']
print(f'Top SKU: {top_sku} | Periods: {len(sku_series)}')
try:
    arima = ARIMA(sku_series.values, order=(1,1,1)).fit()
    arima_pred = arima.forecast(steps=3)
    print(f'ARIMA 3-step forecast: {arima_pred.round(0)}')
except Exception as e:
    print(f'ARIMA error: {e}')

## 4. Forecast vs Actual Chart

In [ ]:
test_plot = test.copy()
test_plot['xgb_pred'] = preds
sku_plot = test_plot[test_plot['sku_id']==top_sku]
fig = go.Figure()
fig.add_scatter(x=sku_plot['period'].astype(str), y=sku_plot['net_units'], name='Actual', line=dict(color='#4B8BBE', width=2))
fig.add_scatter(x=sku_plot['period'].astype(str), y=sku_plot['xgb_pred'].clip(lower=0), name='XGBoost', line=dict(color='#4EC994', dash='dash', width=2))
fig.update_layout(title=f'Forecast vs Actual — {top_sku}', template='plotly_dark', xaxis_tickangle=-30)
fig.show()

## 5. MLflow Experiment Tracking

In [ ]:
import mlflow
from config import settings
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment(settings.mlflow_experiment)
with mlflow.start_run(run_name='notebook_xgboost_eval'):
    mlflow.log_params({'n_estimators':300,'max_depth':5,'learning_rate':0.05})
    mlflow.log_metrics({'test_rmse':round(rmse,2),'test_mape':round(mape,2)})
    print(f'Logged to MLflow — RMSE: {rmse:.2f}, MAPE: {mape:.2f}%')

## 6. Feature Importance

In [ ]:
imp = pd.Series(model.feature_importances_, index=FEAT_COLS).sort_values(ascending=True)
fig2 = px.bar(imp.reset_index(), x=0, y='index', orientation='h',
              title='XGBoost Feature Importance', template='plotly_dark', labels={'0':'Importance','index':'Feature'})
fig2.show()